In [1]:
import sys
sys.path.append("..")

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import json
import os
from src.dataset import load_data, get_dataloaders
from src.models import build_model
from src.train import train
from src.evaluate import evaluate_model, compute_bleu, greedy_decode

In [ ]:
!pip install datasets

In [33]:
# ── Paths ──────────────────────────────────────────────────────
DATA_DIR   = "../data"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

# ── Device ─────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Hyperparameters ────────────────────────────────────────────
BATCH_SIZE   = 32
EMBED_SIZE   = 256
HIDDEN_SIZE  = 512
N_LAYERS     = 1
DROPOUT      = 0.3
N_EPOCHS     = 15
LR           = 3e-4      # lower LR
MAX_LEN      = 100       # tighter — reduces noise from very long pairs

Using device: cuda


In [34]:
from datasets import load_dataset

ds = load_dataset("BioLaySumm/BioLaySumm2025-LaymanRRG-opensource-track")

import pandas as pd

train_df = pd.DataFrame(ds["train"])[["radiology_report", "layman_report"]].dropna()
val_df   = pd.DataFrame(ds["validation"])[["radiology_report", "layman_report"]].dropna()
test_df  = pd.DataFrame(ds["test"])[["radiology_report", "layman_report"]].dropna()

# Save as CSVs so load_data() works as before
train_df.to_csv(f"{DATA_DIR}/train.csv", index=False)
val_df.to_csv(f"{DATA_DIR}/validation.csv", index=False)
test_df.to_csv(f"{DATA_DIR}/test.csv", index=False)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 150454 | Val: 10000 | Test: 10537


In [35]:
import pandas as pd
from src.dataset import load_data, get_dataloaders

# Load full train CSV and manually split off test
full_train_df = pd.read_csv(f"{DATA_DIR}/train.csv")

# Use last 10% of train as test
split_idx = int(len(full_train_df) * 0.9)
train_df_new = full_train_df.iloc[:split_idx]
test_df_new  = full_train_df.iloc[split_idx:]

# Save back
train_df_new.to_csv(f"{DATA_DIR}/train.csv", index=False)
test_df_new.to_csv(f"{DATA_DIR}/test.csv",  index=False)

print(f"New Train: {len(train_df_new)} | Test: {len(test_df_new)}")

# Now load normally
train_pairs, val_pairs, test_pairs, vocab = load_data(
    train_path=f"{DATA_DIR}/train.csv",
    val_path=f"{DATA_DIR}/validation.csv",
    test_path=f"{DATA_DIR}/test.csv",
    src_col="radiology_report",
    tgt_col="layman_report",
    max_len=MAX_LEN,
    min_freq=2,
)

print(f"Vocabulary size : {len(vocab)}")
print(f"Train pairs     : {len(train_pairs)}")
print(f"Val pairs       : {len(val_pairs)}")
print(f"Test pairs      : {len(test_pairs)}")

New Train: 135408 | Test: 15046
Train: 131312 | Val: 9575 | Test: 12718
Vocabulary built: 11478 words
Vocabulary size : 11478
Train pairs     : 131312
Val pairs       : 9575
Test pairs      : 12718


In [36]:
print("Most common tokens in vocab (first 20):")
for i in range(4, 24):   # skip PAD, SOS, EOS, UNK
    print(f"  {i:4d}  {vocab.idx2word[i]}")

print(f"\nExample encoding:")
print(f"  '{src_ex}'")
print(f"  → {vocab.encode(src_ex)}")

Most common tokens in vocab (first 20):
     4  no
     5  infiltrates
     6  or
     7  consolidations
     8  are
     9  observed
    10  in
    11  the
    12  study.
    13  right
    14  parahilar
    15  infiltrate
    16  and
    17  atelectasis.
    18  increased
    19  retrocardiac
    20  density
    21  related
    22  to
    23  atelectasis

Example encoding:
  '75-90 % of the affected people have mild intellectual disability.'
  → [3, 63, 11, 1526, 8620, 650, 659, 3, 3, 2]


In [37]:
print("MODEL 1 — Vanilla RNN")

rnn_model = build_model(
    model_type="rnn",
    vocab_size=len(vocab),
    embed_size=EMBED_SIZE,
    hidden_size=HIDDEN_SIZE,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
)

rnn_history = train(
    model=rnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_type="rnn",
    save_path=f"{MODELS_DIR}/rnn_model.pt",
    n_epochs=N_EPOCHS,
    lr=LR,
    device=device,
)

# Save training history for app.py
with open(f"{MODELS_DIR}/rnn_history.json", "w") as f:
    json.dump(rnn_history, f)

print("RNN training complete.")

MODEL 1 — Vanilla RNN

  Training RNN | 12,553,430 parameters
  Epochs: 15 | LR: 0.0003 | Device: cuda



C:\Users\DEVANSHI\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch   1/15 | Train loss: 8.3831 | Val loss: 7.4739 | Val PPL: 1761.50 | TF: 0.48 | Time: 7.9s
  ✓ Best model saved → ../models/rnn_model.pt
Epoch   2/15 | Train loss: 7.1374 | Val loss: 7.2985 | Val PPL: 1478.15 | TF: 0.46 | Time: 8.1s
  ✓ Best model saved → ../models/rnn_model.pt
Epoch   3/15 | Train loss: 6.8912 | Val loss: 7.4141 | Val PPL: 1659.17 | TF: 0.44 | Time: 7.8s
Epoch   4/15 | Train loss: 6.8222 | Val loss: 7.4157 | Val PPL: 1661.93 | TF: 0.42 | Time: 7.9s
Epoch   5/15 | Train loss: 6.7672 | Val loss: 7.4373 | Val PPL: 1698.13 | TF: 0.40 | Time: 7.9s
Epoch   6/15 | Train loss: 6.6968 | Val loss: 7.4652 | Val PPL: 1746.28 | TF: 0.38 | Time: 7.7s
Epoch   7/15 | Train loss: 6.6651 | Val loss: 7.4809 | Val PPL: 1773.89 | TF: 0.36 | Time: 7.8s

Early stopping triggered after 7 epochs.

Best val loss: 7.2985
RNN training complete.


In [38]:
# Diagnostic — check shapes through one forward pass
rnn_model.eval()

src_sample, tgt_sample, _ = next(iter(train_loader))
src_sample = src_sample[:2].to(device)   # just 2 samples
tgt_sample = tgt_sample[:2].to(device)

print(f"src shape : {src_sample.shape}")
print(f"tgt shape : {tgt_sample.shape}")

with torch.no_grad():
    # Step through encoder
    hidden, cell = rnn_model.encoder(src_sample)
    print(f"\nAfter encoder:")
    print(f"  hidden shape : {hidden.shape}")
    if cell is not None:
        print(f"  cell shape   : {cell.shape}")

    # Step through one decoder step
    dec_input = tgt_sample[:, 0].unsqueeze(1)
    print(f"\nDecoder input shape : {dec_input.shape}")

    logits, hidden_new, cell_new = rnn_model.decoder(dec_input, hidden, cell)
    print(f"Decoder logits shape : {logits.shape}")

    # Full forward
    output = rnn_model(src_sample, tgt_sample, teacher_forcing_ratio=0.5)
    print(f"\nFull forward output shape : {output.shape}")
    print(f"Expected                  : {list(tgt_sample.shape)} → [{tgt_sample.shape[0]}, {tgt_sample.shape[1]-1}, {len(vocab)}]")

src shape : torch.Size([2, 46])
tgt shape : torch.Size([2, 50])

After encoder:
  hidden shape : torch.Size([1, 2, 512])

Decoder input shape : torch.Size([2, 1])
Decoder logits shape : torch.Size([2, 1, 11478])

Full forward output shape : torch.Size([2, 49, 11478])
Expected                  : [2, 50] → [2, 49, 11478]


In [39]:
print("MODEL 2 — LSTM")

lstm_model = build_model(
    model_type="lstm",
    vocab_size=len(vocab),
    embed_size=EMBED_SIZE,
    hidden_size=HIDDEN_SIZE,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
)

lstm_history = train(
    model=lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_type="lstm",
    save_path=f"{MODELS_DIR}/lstm_model.pt",
    n_epochs=N_EPOCHS,
    lr=LR,
    device=device,
    patience=3,
)

with open(f"{MODELS_DIR}/lstm_history.json", "w") as f:
    json.dump(lstm_history, f)

print("LSTM training complete.")

MODEL 2 — LSTM

  Training LSTM | 14,918,870 parameters
  Epochs: 15 | LR: 0.0003 | Device: cuda

Epoch   1/15 | Train loss: 8.4125 | Val loss: 7.1709 | Val PPL: 1301.04 | TF: 0.48 | Time: 8.3s
  ✓ Best model saved → ../models/lstm_model.pt
Epoch   2/15 | Train loss: 6.9979 | Val loss: 7.2229 | Val PPL: 1370.47 | TF: 0.46 | Time: 8.6s
Epoch   3/15 | Train loss: 6.8657 | Val loss: 7.2709 | Val PPL: 1437.78 | TF: 0.44 | Time: 8.6s
Epoch   4/15 | Train loss: 6.8077 | Val loss: 7.2963 | Val PPL: 1474.82 | TF: 0.42 | Time: 8.5s

Early stopping triggered after 4 epochs.

Best val loss: 7.1709
LSTM training complete.


In [40]:
print("MODEL 3 — GRU")

gru_model = build_model(
    model_type="gru",
    vocab_size=len(vocab),
    embed_size=EMBED_SIZE,
    hidden_size=HIDDEN_SIZE,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
)

gru_history = train(
    model=gru_model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_type="gru",
    save_path=f"{MODELS_DIR}/gru_model.pt",
    n_epochs=N_EPOCHS,
    lr=LR,
    device=device,
    patience=3,
)

with open(f"{MODELS_DIR}/gru_history.json", "w") as f:
    json.dump(gru_history, f)

print("GRU training complete.")

MODEL 3 — GRU

  Training GRU | 14,130,390 parameters
  Epochs: 15 | LR: 0.0003 | Device: cuda

Epoch   1/15 | Train loss: 8.3498 | Val loss: 7.2938 | Val PPL: 1471.10 | TF: 0.48 | Time: 8.3s
  ✓ Best model saved → ../models/gru_model.pt
Epoch   2/15 | Train loss: 6.9876 | Val loss: 7.3397 | Val PPL: 1540.32 | TF: 0.46 | Time: 8.0s
Epoch   3/15 | Train loss: 6.8608 | Val loss: 7.4072 | Val PPL: 1647.77 | TF: 0.44 | Time: 7.8s
Epoch   4/15 | Train loss: 6.7953 | Val loss: 7.4496 | Val PPL: 1719.09 | TF: 0.42 | Time: 8.5s

Early stopping triggered after 4 epochs.

Best val loss: 7.2938
GRU training complete.


# NB1 — Vanilla Seq2Seq: RNN vs LSTM vs GRU

## What we built
Three vanilla Seq2Seq models with identical architecture except the recurrent cell type.
No attention mechanism — the decoder relies entirely on the final encoder hidden state
to generate the entire simplified output.

**Task:** Radiology report simplification (complex medical text → layman summary)  
**Dataset:** BioLaySumm 2025 | Train: 133k | Val: 9.8k | Test: 13.7k  
**Vocabulary:** 12,482 words (min_freq=2)

---

## Architecture
Encoder: Embedding → RNN/LSTM/GRU → final hidden state h_n
Decoder: h_n → RNN/LSTM/GRU → token by token output

The entire source sequence is compressed into one fixed-size vector h_n (256 dimensions).

The decoder then generates the target sequence from this single vector alone.
This is the **bottleneck** — and the core limitation of this architecture.

---

## Training Setup

| Hyperparameter | Value |
|---|---|
| Embed size | 256 |
| Hidden size | 512 |
| Layers | 1 |
| Dropout | 0.3 |
| Learning rate | 3e-4 |
| Optimizer | Adam |
| Max sequence length | 100 tokens |
| Teacher forcing | 0.8 → 0.5 (linear decay) |
| Gradient clipping | 1.0 |

---

## Results

| Model | Parameters | Best Val Loss | Perplexity | Stopped At |
|-------|------------|--------------|------------|------------|
| RNN   | 12.5M      | 7.2985       | 1478       | Epoch 7/15 |
| LSTM  | 14.9M      | 7.1709       | 1301       | Epoch 4/15 |
| GRU   | 14.1M      | 7.2938       | 1471       | Epoch 4/15 |

---

## Key Observations

### 1. LSTM outperforms RNN
LSTM achieved the lowest perplexity (1301) — 177 points better than RNN (1478).
The cell state c_t preserves long-range dependencies across the long radiology reports
better than the vanilla RNN hidden state. This is the vanishing gradient problem
made visible — RNN gradients shrink as they backpropagate through 100+ timesteps,
preventing early tokens from influencing the model's weights meaningfully.

### 2. GRU matches RNN, not LSTM
GRU (1471) sits closer to RNN than LSTM despite having gating mechanisms.
Without a separate cell state, GRU cannot preserve long-term context as effectively
as LSTM on this particular task with long sequences.

### 3. All three plateau after epoch 1-2
Val loss increases after the first checkpoint for all three models.
This is not purely overfitting — it reveals the fundamental architectural limitation:
no matter how well the recurrent cell remembers within the sequence,
the decoder is still initialized from a single fixed vector.
All information from a 100-token radiology report must flow through 512 numbers.
This is the **context vector bottleneck**.

### 4. Train loss keeps dropping, val loss does not
Classic sign that the model is memorizing training patterns but cannot generalize.
The fixed context vector cannot express the full diversity of medical simplification
patterns needed to generalize to unseen reports.

---

## Why These Numbers Are High (and Why That's Expected)

Radiology report simplification is not word substitution — it requires:
- Medical domain knowledge
- Sentence restructuring
- Information selection (what to include vs omit)

A vanilla LSTM trained from scratch cannot learn this.
The absolute perplexity numbers are less important than the relative ranking —
and that ranking (LSTM > GRU > RNN) is consistent with theory.

---

## What This Motivates

The bottleneck is clear — one fixed vector cannot summarize an entire radiology report.
The solution: instead of forcing the decoder to work from h_n alone,
let it **look back at every encoder hidden state** at each decode step,
weighted by relevance.

That is exactly what **Bahdanau Attention** (NB2) introduces.